# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya

This notebook provides a guided template for loading and exploring the FAIR$^2$ dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is described by a Croissant schema (JSON-LD):

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure mlcroissant is installed (uncomment and run if needed)
!pip install mlcroissant

## 1. Data Loading
Load metadata and available record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant JSON-LD schema URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset (includes schema and references to records/record sets)
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # This is the Croissant Dataset metadata object

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Date published: {metadata.datePublished}")
print(f"Temporal coverage: {metadata.temporalCoverage}")
print(f"License: {metadata.license}")


## 2. Data Overview
Let's list the available record sets and their fields. 
We will reference everything using their Croissant `@id` fields,
which ensures all code and analysis are reliably mapped to the schema.

In [ ]:
# List all record set ids and their fields with @id references
print("Available record sets:")
record_sets = [rs for rs in dataset.record_sets()]
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    if 'field' in rs:
        if isinstance(rs['field'], list):
            for field in rs['field']:
                if isinstance(field, dict):
                    print(f"    Field @id: {field['@id']} | Name: {field.get('name', '<no name>')}")
                else:
                    print(f"    Field @id: {field}")
        else:
            print(f"    Field @id: {rs['field']}")
    else:
        print("    No explicit fields listed.")

## 3. Data Extraction
Let's extract tables from the available record sets into Pandas DataFrames.
We reference record sets by their `@id`, and load the whole set.

🔑 **Note**: You'll need to scan the previous cell's output to choose the record set `@id`s you want to analyze. For this notebook, we'll illustrate by trying to load all available record sets.

In [ ]:
dataframes = {}
# record_sets variable defined above (as a list of dicts)
for rs in record_sets:
    record_set_id = rs['@id']
    print(f"\nLoading records from RecordSet @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame with shape {df.shape} and columns: {df.columns.tolist()}")
        else:
            print("No records found for this record set.")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

if len(dataframes) > 0:
    # Pick the first (or any) record set for display
    display_id = list(dataframes.keys())[0]
    print(f"\nPreview of first 5 records from RecordSet {display_id}:")
    display(dataframes[display_id].head())
else:
    print("No record sets were loaded into DataFrames.")

## 4. Exploratory Data Analysis (EDA)

- Filter records based on numeric fields
- Normalize numeric columns
- Optionally, group by categorical fields

All referenced columns and record sets use their `@id`s, as required.

In [ ]:
# Pick a record set and numeric/categorical fields by @id (update as needed).
if dataframes:
    # Choose any available record set (user may update this selection)
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    print(f"Working with RecordSet @id: {record_set_id}")
    print(f"Available columns: {df.columns.tolist()}")

    # Try to auto-select a numeric field and a group field
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
            group_field_id = col
            break

    if numeric_field_id is None:
        print("No numeric field found for EDA. Skipping.")
    else:
        # Remove NaNs to avoid filtering errors
        df = df.dropna(subset=[numeric_field_id])
        threshold = df[numeric_field_id].mean() if not df[numeric_field_id].empty else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())
            / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id}:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouped analysis
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
            print(grouped_df.head())
else:
    print("No DataFrames to analyze. Please rerun previous cell after loading data.")

## 5. Visualization
Visualize distributions or relationships between fields (referencing fields by `@id`).

In [ ]:
# Simple visualizations using matplotlib or seaborn
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = list(dataframes.values())[0]
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if numeric_cols:
        field_id = numeric_cols[0]
        plt.figure(figsize=(7, 4))
        sns.histplot(df[field_id].dropna(), kde=True)
        plt.title(f'Distribution of {field_id}')
        plt.xlabel(field_id)
        plt.show()

        # If there's a categorical field, plot its relation to the numeric
        group_fields = [c for c in df.columns if df[c].dtype == object]
        if group_fields:
            group_field = group_fields[0]
            plt.figure(figsize=(9,5))
            sns.boxplot(x=df[group_field], y=df[field_id])
            plt.xticks(rotation=35)
            plt.title(f'{field_id} by {group_field}')
            plt.xlabel(group_field)
            plt.ylabel(field_id)
            plt.show()
    else:
        print("No numeric columns to visualize.")
else:
    print("No DataFrames available for visualization.")

## 6. Conclusion

- This notebook illustrated accessing and exploring a dataset described by a Croissant schema using the `mlcroissant` library.
- All record sets, fields, and analysis steps referenced elements by their `@id` to ensure full traceability to the dataset schema.
- Next steps could include more domain-specific EDA, modeling, or merging with external sources.